1. Выберите датасет

Датасет взят с kaggle: https://www.kaggle.com/datasets/yasserh/wine-quality-dataset/data

2. Первичный анализ данных

In [ ]:
import pandas as pd

df = pd.read_csv('WineQT.csv')
print("--- Статистика числовых признаков ---")
print(df.describe())
print("--- Пропущенные значения в колонках ---")
null_counts = df.isnull().sum()
print(null_counts[null_counts > 0])
print("Количество вин по оценкам качества:")
print(df['quality'].value_counts().sort_index())

1) размер датасета: 1143 объекта(строк), 13 признаков(столбцов)
2) типы признаков: числовые ( Primary Key = Id, 12 признаков(кислотность, уровень сахара и др.) )
3) наличие пропусков: нету
4) распределение классов: есть дисбаланс (очень сильно преобладают оценки 5 и 6)
5) возможные проблемы в данных:
- значения density и total sulfur dioxide отличаются вплоть до сотни. Это сильно уменьшит вес первого, и сильно увеличит вес второго признаков
- кислотность и pH почти намертво связаны между собой, это может повлиять на распределение весов

3. Подготовка данных


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
# важно отметить, что текста или пропусков в таблице нету => OHE и замены пучтышек не требуются.

# 2. Удаление ненужных столбцов
# Id — это просто порядковый номер, он только запутает модель
X = df.drop(['quality', 'Id'], axis=1)
y = df['quality'] # Это то, что должно быть предсказано, поэтому тоже не надо

# 3. Разделение на обучающую (train) и тестовую (test) выборки
# random_state=42 нужен, чтобы разделение было одинаковым при каждом запуске
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Масштабирование признаков (Standardization)
scaler = StandardScaler()

# Обучаем на тренировочных данных и сразу трансформируем их
X_train_scaled = scaler.fit_transform(X_train)

# Тестовые данные только трансформируем (используем параметры, полученные на train)
X_test_scaled = scaler.transform(X_test)

# Проверка результата
print(f"Размер обучающей выборки: {X_train_scaled.shape}")
print(f"Размер тестовой выборки: {X_test_scaled.shape}")
print("\nПример первых 2 строк отмасштабированных данных:")
print(X_train_scaled[:2])

- почему масштабирование важно для KNN:
Модель должна понять: какие признаки действительно значительно влияют на расчеты. Именно для этого все сводиться к опред. диапазону.

- почему нельзя подбирать параметры на тестовой выборке:
В таком случае возможно переобучение.

4. Обучение KNN

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# Определяем наборы параметров для исследования
ks = [3, 5, 7, 9, 11, 13, 15]
best_acc = 0
best_params = ""
weights_options = ['uniform', 'distance']
metrics = ['euclidean', 'manhattan']

print(f"{'K':<3} | {'Weights':<10} | {'Metric':<12} | {'Accuracy':<8}")
print("-" * 42)

# Цикл для перебора всех комбинаций
for k in ks:
    for weight in weights_options:
        for metric in metrics:
            # Создаем модель с текущими параметрами
            knn = KNeighborsClassifier(n_neighbors=k, weights=weight, metric=metric)

            # Обучаем на масштабированных данных из 3-го пункта
            knn.fit(X_train_scaled, y_train)

            # Делаем предсказание на тестовой выборке
            y_pred = knn.predict(X_test_scaled)

            # Считаем точность
            acc = accuracy_score(y_test, y_pred)
            if acc > best_acc:
                best_acc = acc
                best_params = f"K={k}, Weights='{weight}', Metric='{metric}'"
            print(f"{k:<3} | {weight:<10} | {metric:<12} | {acc:.4f}")
print("-" * 50)
print("Лучшая модель")
print(f"Точность (Accuracy): {best_acc:.4f}")
print(f"Параметры: {best_params}")

Из таблицы видно, что оптимальный выбор k = 11, при учитывании весов ч/з расстояние между соседями

5. Подбор гиперпараметров

In [ ]:
from sklearn.model_selection import GridSearchCV

# 1. Задаем сетку параметров
param_grid = {
    'n_neighbors': [3, 5, 7, 9, 11, 13, 15, 21],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

# 2. Настраиваем поиск с кросс-валидацией на 5 фолдов
grid_search = GridSearchCV(
    KNeighborsClassifier(),
    param_grid,
    cv=5,
    scoring='accuracy'
)

# 3. Обучаем (теперь модель сама внутри себя делит X_train_scaled на части)
grid_search.fit(X_train_scaled, y_train)

# 4. Выводим результаты
print("РЕЗУЛЬТАТЫ КРОСС-ВАЛИДАЦИИ:")
print("-" * 30)
print(f"Лучшие параметры: {grid_search.best_params_}")
print(f"Лучшая средняя точность (Mean CV Accuracy): {grid_search.best_score_:.4f}")

# 5. Финальная проверка лучшей модели на отложенном тесте
final_model = grid_search.best_estimator_
final_acc = final_model.score(X_test_scaled, y_test)
print(f"Точность на тестовой выборке: {final_acc:.4f}")

Выводы:

Исследование модели KNN
- Число соседей (k): Слишком малое k вело к переобучению, а слишком большое — к потере точности из-за избыточного усреднения.
- Параметр weights='distance' показал лучший результат, чем uniform. Это доказывает, что при классификации качества вина более похожие образцы должны иметь больший приоритет.
- Метрика Manhattan показала себя чуть стабильнее Евклидовой.

Кросс-валидация и подбор параметров
- GridSearchCV нашел «золотую середину» параметров, которая обеспечивает лучшую обобщающую способность модели.

Итоговая точность
- Почти 70% - вполне неплохой результат, учитывая кол-во классов в датасете.

Пить ли вино?
- Да, но немного и хорошее.